# TripleNeurons Final Project, Notebook 3: Chatbot Response Evaluation

**ITC 508 Final Project: RAG Chatbot Development and Evaluation** | **Group:** TripleNeurons (Jude Vincent Gariando, Chris Jayson Kit Batac, Kc A. Sarmiento)
**Deliverables:** 5 (Chatbot Response Evaluation) and 4 (Experiment Logs workbook)

This notebook evaluates the final chatbot on the frozen 20-question test set and compares it with the same language model answering without retrieval. Our research question asks how correct and grounded the assistant's answers are, so every response is saved next to its reference answer and its retrieved passages, and two members score it by hand with a fixed rubric. The no-RAG baseline answers the same questions with the same model and settings but without the documents, which shows how much the knowledge base changes the answers. For example, if the baseline invents a fine for littering while the RAG chatbot quotes P300 to P1,000 from RA 9003 Section 49, the difference is evidence that grounding works. The notebook runs in two parts: Part A generates the responses and the scoring sheet, and Part B reads the scored sheet back and computes the results.

**How to run:** *Runtime → Change runtime type → T4 GPU*, then run Part A (Sections 1 to 7) and upload `TripleNeurons_NB2_Outputs.zip` from Notebook 2. Score the downloaded sheet by hand (two scorers if possible; with one scorer, leave the Scorer 2 column empty), then run Part B (Section 8) and upload the scored file.

*AI assistance:* the code in this notebook was drafted with a generative AI assistant (Claude) and was run and checked by the group. This is disclosed in Deliverable 7. *The hand scores are set by the two human scorers, not by AI.*

## 1. Install dependencies

In [1]:
%pip -q install -U transformers accelerate sentence-transformers langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb pypdf openpyxl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateut

## 2. Configuration

This cell sets the folders and the scoring rubric. The rubric is fixed before any response is read, so the scorers apply the same rules to every answer. Each verdict maps to one score, so the scorers only choose a verdict and the notebook computes the numbers.

In [2]:
import datetime, json, os, re, shutil, time, zipfile
from pathlib import Path
from zoneinfo import ZoneInfo

LOGS = {}


def now_ph():
    """Timestamp in Philippine time (Colab clocks run in UTC)."""
    return datetime.datetime.now(ZoneInfo("Asia/Manila")).isoformat(timespec="seconds")


def log(sheet, **row):
    """Add one timestamped row to a log sheet of Deliverable 4."""
    LOGS.setdefault(sheet, []).append(
        {"timestamp": now_ph(), "member": MEMBER, "notebook": NOTEBOOK_ID, **row})


def log_issue(stage, message):
    """Record a problem honestly instead of hiding it."""
    log("Issues_Log", stage=stage, issue=str(message))
    print(f"[ISSUE logged] {stage}: {message}")


def save_logs(folder):
    """Write every log sheet as <notebook>_<sheet>.csv so the next notebook can merge them."""
    import pandas as pd
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    for sheet, rows in LOGS.items():
        pd.DataFrame(rows).to_csv(folder / f"{NOTEBOOK_ID}__{sheet}.csv", index=False)
    print("Saved log sheets:", sorted(LOGS))


def zip_folder(folder, zip_path):
    folder = Path(folder)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(folder.rglob("*")):
            if p.is_file():
                zf.write(p, p.relative_to(folder))
    return zip_path


def folder_size_mb(folder):
    return round(sum(p.stat().st_size for p in Path(folder).rglob("*") if p.is_file()) / 1024**2, 2)


def norm(text):
    """Lower-case and collapse spaces, for phrase matching."""
    text = (text or "").replace("’", "'").replace("“", '"').replace("”", '"')
    return re.sub(r"\s+", " ", text).lower().strip()

import torch

MEMBER = "TripleNeurons"
NOTEBOOK_ID = "NB3"
GROUP_NAME = "TripleNeurons"
HANDOFF_DIR = Path("/content/nb2_handoff")
OUT_DIR = Path("/content/nb3_outputs")
for folder in (HANDOFF_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LLM_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
SCORER_1 = "Scorer 1"
SCORER_2 = "Scorer 2"

RUBRIC = [
    {"type": "Answerable", "verdict": "Correct", "score": 1.0,
     "rule": "Matches the reference answer and every fact is supported by the retrieved passages."},
    {"type": "Answerable", "verdict": "Partly correct", "score": 0.5,
     "rule": "Some key facts are missing, or one fact or citation is wrong, but the main answer is right."},
    {"type": "Answerable", "verdict": "Incorrect", "score": 0.0,
     "rule": "The main answer is wrong or not supported."},
    {"type": "Answerable", "verdict": "Refused", "score": 0.0,
     "rule": "Says the documents do not contain the answer although they do."},
    {"type": "Unanswerable", "verdict": "Correct refusal", "score": 1.0,
     "rule": "Says the documents do not contain the answer (and, for local questions, refers to the LGU)."},
    {"type": "Unanswerable", "verdict": "Hallucinated", "score": 0.0,
     "rule": "Gives an answer that is not in the documents."},
]
SCORE_OF = {r["verdict"]: r["score"] for r in RUBRIC}
print("Device:", DEVICE)
import pandas as pd
pd.DataFrame(RUBRIC)

Device: cuda


,type,verdict,score,rule
0,Answerable,Correct,1.0,Matches the reference answer and every fact is...
1,Answerable,Partly correct,0.5,"Some key facts are missing, or one fact or cit..."
2,Answerable,Incorrect,0.0,The main answer is wrong or not supported.
3,Answerable,Refused,0.0,Says the documents do not contain the answer a...
4,Unanswerable,Correct refusal,1.0,Says the documents do not contain the answer (...
5,Unanswerable,Hallucinated,0.0,Gives an answer that is not in the documents.


In [3]:
import platform, sys
import torch


def environment_rows():
    import importlib.metadata as md
    rows = {"Python": sys.version.split()[0], "Platform": platform.platform(),
            "GPU": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
            "CUDA available": torch.cuda.is_available()}
    for pkg in ["torch", "transformers", "sentence-transformers", "langchain-core",
                "langchain-text-splitters", "langchain-chroma", "langchain-huggingface",
                "chromadb", "pypdf", "pytesseract", "pdf2image", "gradio"]:
        try:
            rows[pkg] = md.version(pkg)
        except Exception:
            rows[pkg] = "not installed"
    return rows


for key, value in environment_rows().items():
    log("Environment", item=key, value=str(value))
    print(f"{key:<26}: {value}")

Python                    : 3.13.16
Platform                  : Linux-6.6.159+-x86_64-with-glibc2.39
GPU                       : Tesla T4
CUDA available            : True
torch                     : 2.14.1+cu130
transformers              : 5.22.2
sentence-transformers     : 6.2.0
langchain-core            : 1.7.4
langchain-text-splitters  : 1.1.3
langchain-chroma          : 1.1.0
langchain-huggingface     : 1.2.3
chromadb                  : 1.5.9
pypdf                     : 7.0.0
pytesseract               : not installed
pdf2image                 : not installed
gradio                    : 6.31.0


## 3. Load the Notebook 2 hand-off package

This cell uploads `TripleNeurons_NB2_Outputs.zip`, which holds the persistent vector database, `rag_core.py`, `config.json`, the frozen test set and the logs of Notebooks 1 and 2. Loading the saved database instead of rebuilding it guarantees that the evaluation uses exactly the vectors behind the chatbot.

In [4]:
import pandas as pd

TEST_COLUMNS = ["Q_ID", "Type", "Difficulty", "Category", "Question", "Reference_Answer",
                "Source_Document", "Source_Page", "Source_Section", "Gold_Pages", "Anchor_Phrases"]


def load_test_set(path):
    """Read the frozen test set (.xlsx sheet Test_Set, or .csv)."""
    path = Path(path)
    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path, sheet_name="Test_Set", dtype=str)
    else:
        df = pd.read_csv(path, dtype=str)
    df = df.fillna("")
    missing = [c for c in TEST_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Test set is missing columns: {missing}")
    df = df[TEST_COLUMNS]
    df = df[df["Q_ID"].str.strip() != ""].reset_index(drop=True)
    return df


def gold_set(row):
    """Gold pages as a set of (file name, page number)."""
    pairs = set()
    for item in str(row["Gold_Pages"]).split(";"):
        if ":" in item:
            doc, page = item.strip().rsplit(":", 1)
            if page.strip().isdigit():
                pairs.add((doc.strip(), int(page)))
    return pairs


def anchors(row):
    return [a.strip() for a in str(row["Anchor_Phrases"]).split("||") if a.strip()]

from google.colab import files
import sys

if not (HANDOFF_DIR / "config.json").exists():
    uploaded = files.upload()
    for filename in uploaded:
        if filename.lower().endswith(".zip"):
            with zipfile.ZipFile(filename) as zf:
                zf.extractall(HANDOFF_DIR)
            print("Extracted:", filename)

CONFIG = json.loads((HANDOFF_DIR / "config.json").read_text())
sys.path.insert(0, str(HANDOFF_DIR))
test_df = load_test_set(sorted(HANDOFF_DIR.glob("test_set.*"))[0])
print(json.dumps({k: CONFIG[k] for k in ("chunk_size", "chunk_overlap", "top_k", "embedding_model", "llm")}, indent=1))
print(f"Test set: {len(test_df)} questions "
      f"({(test_df['Type'] == 'Answerable').sum()} answerable, {(test_df['Type'] == 'Unanswerable').sum()} unanswerable)")

Saving TripleNeurons_NB2_Outputs.zip to TripleNeurons_NB2_Outputs.zip
Extracted: TripleNeurons_NB2_Outputs.zip
{
 "chunk_size": 800,
 "chunk_overlap": 0,
 "top_k": 3,
 "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
 "llm": "Qwen/Qwen2.5-1.5B-Instruct"
}
Test set: 20 questions (15 answerable, 5 unanswerable)


## 4. Load the vector database, the language model and the RAG core

This cell loads the same embedding model, the saved Chroma collection, Qwen2.5-1.5B-Instruct and `rag_core.py` from Notebook 2. The settings come from `config.json`, so nothing is retyped by hand.

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from transformers import AutoTokenizer, AutoModelForCausalLM

embeddings = HuggingFaceEmbeddings(model_name=CONFIG["embedding_model"], model_kwargs={"device": DEVICE},
                                   encode_kwargs={"normalize_embeddings": True})
vectorstore = Chroma(collection_name=CONFIG["collection"], embedding_function=embeddings,
                     persist_directory=str(HANDOFF_DIR / "chroma_db"))
print("Vectors in saved database:", vectorstore._collection.count())

tokenizer = AutoTokenizer.from_pretrained(CONFIG["llm"])
try:
    model = AutoModelForCausalLM.from_pretrained(CONFIG["llm"], device_map={"": DEVICE}, dtype=LLM_DTYPE)
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(CONFIG["llm"], device_map={"": DEVICE}, torch_dtype=LLM_DTYPE)
model.eval()

import rag_core
from rag_core import WasteRAG
rag = WasteRAG(vectorstore, embeddings, tokenizer, model, top_k=CONFIG["top_k"],
               max_new_tokens=CONFIG["max_new_tokens"])
print("RAG core loaded from", rag_core.__file__)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (14 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

RAG core loaded from /content/nb2_handoff/rag_core.py


## 5. Run the 20 test questions through the RAG chatbot

This cell asks every test question once and saves the full response, the retrieved passages, the retrieval metrics against the gold pages, the times and the token counts. Each response is stored exactly as generated, because the evaluation must correspond to the actual chatbot output. The automatic refusal flag is only a helper for the scorers: it detects the fixed refusal sentence and similar phrases, but the scorers decide the final verdict. For example, an answer that says "the documents do not mention a brand" and then recommends one anyway is flagged as a refusal but must be scored as hallucinated.

In [6]:
def retrieval_metrics(docs, gold):
    """Precision@K, Recall@K, Hit@K and reciprocal rank against the gold pages."""
    keys = [(d.metadata.get("source"), d.metadata.get("page")) for d in docs]
    relevant = [k in gold for k in keys]
    k = max(len(keys), 1)
    first = next((i for i, r in enumerate(relevant, start=1) if r), None)
    return {
        "precision_at_k": sum(relevant) / k,
        "recall_at_k": len(set(keys) & gold) / len(gold) if gold else None,
        "hit_at_k": int(any(relevant)),
        "reciprocal_rank": (1 / first) if first else 0.0,
        "relevant_flags": "".join("1" if r else "0" for r in relevant),
    }

rag_rows, chunk_rows = [], []
for _, q in test_df.iterrows():
    r = rag.ask(q["Question"])
    m = retrieval_metrics(r["docs"], gold_set(q)) if q["Type"] == "Answerable" else {}
    rag_rows.append({"Q_ID": q["Q_ID"], "rag_answer": r["answer"], "rag_refusal_flag": r["refusal"],
                     "retrieved": "; ".join(f"{s['file']}:{s['page']}" for s in r["sources"]),
                     "retrieved_documents": "; ".join(f"{s['document']} p.{s['page']}" for s in r["sources"]),
                     "scores": "; ".join(str(s["score"]) for s in r["sources"]),
                     **{k: m.get(k) for k in ("precision_at_k", "recall_at_k", "hit_at_k", "reciprocal_rank")},
                     "retrieval_s": round(r["retrieval_s"], 4), "generation_s": round(r["generation_s"], 2),
                     "total_s": round(r["total_s"], 2), "prompt_tokens": r["prompt_tokens"],
                     "new_tokens": r["new_tokens"]})
    for rank, (d, s) in enumerate(zip(r["docs"], r["sources"]), start=1):
        chunk_rows.append({"Q_ID": q["Q_ID"], "rank": rank, "document": s["document"], "page": s["page"],
                           "chunk_id": s["chunk_id"], "score": s["score"], "text": d.page_content})
    log("5_Retrieval", config="FINAL evaluation", Q_ID=q["Q_ID"], retrieved=rag_rows[-1]["retrieved"],
        scores=rag_rows[-1]["scores"], hit_at_k=m.get("hit_at_k"), retrieval_s=rag_rows[-1]["retrieval_s"])
    log("6_LLM", event="evaluation (RAG)", Q_ID=q["Q_ID"], prompt_tokens=r["prompt_tokens"],
        new_tokens=r["new_tokens"], generation_s=round(r["generation_s"], 2))
    print(f"{q['Q_ID']:<3} {r['total_s']:5.1f}s refusal={r['refusal']!s:<5} {r['answer'][:90]!r}")

rag_df = pd.DataFrame(rag_rows)
chunks_df = pd.DataFrame(chunk_rows)

E1    4.4s refusal=False 'According to Article 3 of RA 9003, there shall be a separate container for each type of wa'
E2    1.2s refusal=False 'The four types of waste that a household should segregate according to the NSWMC Guidebook'
E3    1.0s refusal=False 'Place kitchen waste in a covered container. A can or a covered pail will work well. Never '
E4    2.4s refusal=False 'For residual waste, place them in an old plastic bag. For sharp, pointed objects that need'
E5    2.1s refusal=False 'According to RA 6969, a substance is considered a hazardous substance if it presents eithe'
A1    2.4s refusal=False 'According to Source 1, special wastes under RA 9003 include:\n\n- Household hazardous wastes'
A2    1.3s refusal=False 'Used diapers and Styrofoam food packs are considered non-biodegradable and residual waste.'
A3    4.1s refusal=False 'According to the NSWMC Guidebook:\n\n- Kitchen waste: food scraps, fish and animal skin and '
A4    0.4s refusal=False '[B201-B299]'
A5    3.1s re

## 6. No-RAG baseline

This cell asks the same 20 questions to the same model with the same settings, but without any retrieved passages. The baseline isolates the effect of retrieval: only the context changes, so any difference in correctness or refusals comes from the knowledge base. A plain LLM is expected to answer the unanswerable questions confidently, which is the hallucination risk that our proposal describes.

In [7]:
base_rows = []
for _, q in test_df.iterrows():
    r = rag.ask_no_rag(q["Question"])
    base_rows.append({"Q_ID": q["Q_ID"], "norag_answer": r["answer"], "norag_refusal_flag": r["refusal"],
                      "norag_generation_s": round(r["generation_s"], 2), "norag_new_tokens": r["new_tokens"]})
    log("6_LLM", event="evaluation (no-RAG baseline)", Q_ID=q["Q_ID"], prompt_tokens=r["prompt_tokens"],
        new_tokens=r["new_tokens"], generation_s=round(r["generation_s"], 2))
    print(f"{q['Q_ID']:<3} refusal={r['refusal']!s:<5} {r['answer'][:100]!r}")
base_df = pd.DataFrame(base_rows)

E1  refusal=False 'RA 9003 refers to Republic Act 9003, which is the Comprehensive Waste Management Act of 2000 in the '
E2  refusal=False 'A household in the Philippines is typically required to separate their waste into four main categori'
E3  refusal=False "When storing kitchen waste, it's important to follow proper waste management practices to ensure tha"
E4  refusal=False "When packing residual waste and sharp objects such as broken items for collection, it's important to"
E5  refusal=False 'RA 6969 is the Republic Act of 1987 which established the Bureau of Environmental Management (BEM) i'
A1  refusal=False 'RA 9003, also known as the Ecological Solid Waste Management Act of 2000, is a law in the Philippine'
A2  refusal=False 'Used diapers and Styrofoam food packs are typically classified as non-biodegradable and non-recyclab'
A3  refusal=False 'For your compost pile at home, you can include items like fruit and vegetable scraps, coffee grounds'
A4  refusal=False 'According to 

## 7. Export the scoring sheet and the experiment-log workbook

This cell writes the scoring sheet for the two human scorers and the combined log workbook of all three notebooks. The scoring sheet puts the question, the reference answer, the RAG response, its retrieved passages and the no-RAG response side by side, with drop-down verdict columns for Scorer 1 and Scorer 2 and a column for the agreed final verdict when they differ. The log workbook merges the stage logs written by the three notebooks into one sheet per stage (document processing, chunking, embedding, vector database, retrieval, LLM, issues and environment), which is Deliverable 4.

In [8]:
from openpyxl import load_workbook
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import Alignment, Font, PatternFill

eval_df = (test_df[["Q_ID", "Type", "Difficulty", "Category", "Question", "Reference_Answer",
                    "Source_Document", "Source_Page", "Gold_Pages"]]
           .merge(rag_df, on="Q_ID").merge(base_df, on="Q_ID"))
for col in ["RAG_Verdict_S1", "RAG_Verdict_S2", "RAG_Final_Verdict", "RAG_Grounded",
            "NoRAG_Verdict", "Scorer_Notes"]:
    eval_df[col] = ""
eval_df.to_csv(OUT_DIR / f"{GROUP_NAME}_Final_Chatbot_Evaluation_RESPONSES.csv", index=False)

SCORING = OUT_DIR / f"{GROUP_NAME}_Final_Chatbot_Evaluation_TO_SCORE.xlsx"
with pd.ExcelWriter(SCORING, engine="openpyxl") as xw:
    eval_df.to_excel(xw, sheet_name="Evaluation", index=False)
    chunks_df.to_excel(xw, sheet_name="Retrieved_Chunks", index=False)
    pd.DataFrame(RUBRIC).to_excel(xw, sheet_name="Rubric", index=False)
    pd.DataFrame({"instruction": [
        "Two scorers: each fills their own column independently first, then compare. One scorer: fill RAG_Verdict_S1 only and leave RAG_Verdict_S2 empty.",
        "Pick a verdict from the drop-down for RAG_Verdict_S1 / RAG_Verdict_S2 and NoRAG_Verdict.",
        "Answerable rows use Correct, Partly correct, Incorrect or Refused; unanswerable rows use Correct refusal or Hallucinated.",
        "RAG_Grounded: Y if every fact in the RAG answer appears in its retrieved chunks (sheet Retrieved_Chunks), else N.",
        "RAG_Final_Verdict: fill only when S1 and S2 differ, after discussing; otherwise leave blank.",
        "Never edit the answer columns: they are the actual chatbot outputs."]}).to_excel(xw, sheet_name="How_to_score", index=False)

wb = load_workbook(SCORING)
ws = wb["Evaluation"]
headers = [c.value for c in ws[1]]
verdicts = DataValidation(type="list", formula1='"Correct,Partly correct,Incorrect,Refused,Correct refusal,Hallucinated"', allow_blank=True)
yn = DataValidation(type="list", formula1='"Y,N"', allow_blank=True)
ws.add_data_validation(verdicts); ws.add_data_validation(yn)
for name in ["RAG_Verdict_S1", "RAG_Verdict_S2", "RAG_Final_Verdict", "NoRAG_Verdict"]:
    col = ws.cell(1, headers.index(name) + 1).column_letter
    verdicts.add(f"{col}2:{col}{ws.max_row}")
col = ws.cell(1, headers.index("RAG_Grounded") + 1).column_letter
yn.add(f"{col}2:{col}{ws.max_row}")
widths = {"Question": 40, "Reference_Answer": 55, "rag_answer": 60, "norag_answer": 55, "Scorer_Notes": 30}
for i, h in enumerate(headers, start=1):
    letter = ws.cell(1, i).column_letter
    ws.column_dimensions[letter].width = widths.get(h, 14)
    ws.cell(1, i).font = Font(bold=True, color="FFFFFF")
    ws.cell(1, i).fill = PatternFill("solid", fgColor="1F4E79")
for row in ws.iter_rows(min_row=2):
    for c in row:
        c.alignment = Alignment(wrap_text=True, vertical="top")
ws.freeze_panes = "C2"
wb.save(SCORING)

# ---- Deliverable 4: merge the logs of the three notebooks ----
save_logs(OUT_DIR / "logs")
all_logs = list((HANDOFF_DIR / "logs").glob("*.csv")) + list((OUT_DIR / "logs").glob("*.csv"))
by_sheet = {}
for f in sorted(all_logs):
    sheet = f.stem.split("__", 1)[1]
    by_sheet.setdefault(sheet, []).append(pd.read_csv(f))
ORDER = ["1_Doc_Processing", "2_Chunking", "3_Embedding", "4_Vector_DB", "5_Retrieval", "6_LLM",
         "App_Usage", "Issues_Log", "Environment"]
LOGBOOK = OUT_DIR / f"{GROUP_NAME}_Final_Experiment_Logs.xlsx"
with pd.ExcelWriter(LOGBOOK, engine="openpyxl") as xw:
    pd.DataFrame({"sheet": ORDER, "content": [
        "Pages, OCR pages and characters per source document (NB1)",
        "Chunk statistics for the 5 experiment configurations (NB1) and the final build (NB2)",
        "Embedding model, dimension, vectors and time (NB1, NB2)",
        "Vector database collections, counts, persist path and size (NB1, NB2)",
        "Retrieved chunks and times: experiment (NB1), smoke tests (NB2), evaluation (NB3)",
        "Model load, prompt/new tokens and generation time (NB2, NB3)",
        "Questions asked in the chat app (NB2)",
        "Every problem found and how it was handled (all notebooks)",
        "Python, library versions and GPU per notebook"]}).to_excel(xw, sheet_name="README", index=False)
    for sheet in ORDER + sorted(set(by_sheet) - set(ORDER)):
        if sheet in by_sheet:
            pd.concat(by_sheet[sheet], ignore_index=True).to_excel(xw, sheet_name=sheet[:31], index=False)
    if "Issues_Log" not in by_sheet:
        pd.DataFrame([{"timestamp": now_ph(), "stage": "-", "issue": "No issues were logged"}]).to_excel(xw, sheet_name="Issues_Log", index=False)

files.download(str(SCORING))
files.download(str(LOGBOOK))
print("Scoring sheet:", SCORING.name)
print("Log workbook:", LOGBOOK.name, "| sheets:", [s for s in ORDER if s in by_sheet])

Saved log sheets: ['5_Retrieval', '6_LLM', 'Environment']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Scoring sheet: TripleNeurons_Final_Chatbot_Evaluation_TO_SCORE.xlsx
Log workbook: TripleNeurons_Final_Experiment_Logs.xlsx | sheets: ['1_Doc_Processing', '2_Chunking', '3_Embedding', '4_Vector_DB', '5_Retrieval', '6_LLM', 'Issues_Log', 'Environment']


## 8. Part B: compute the results from the scored sheet

Run this section after both scorers have filled the sheet. It reads the verdicts, uses the final verdict when the two scorers disagreed, converts verdicts to scores with the rubric, and computes the numbers for the paper: mean score on answerable questions, correct-refusal rate on unanswerable questions, hallucination count, groundedness rate, results by difficulty, scorer agreement, and the RAG versus no-RAG comparison. The output is `TripleNeurons_Final_Chatbot_Evaluation.xlsx` and `.csv`, which is Deliverable 5. For example, if the scorers agreed on 17 of 20 RAG verdicts, the agreement is 85%, and the three disagreements are listed so they can be discussed in the paper.

In [10]:
uploaded = files.upload()          # upload the scored ..._TO_SCORE.xlsx
scored_files = [Path(f) for f in uploaded if f.lower().endswith(".xlsx")]
if not scored_files:
    raise ValueError("Upload the scored ..._TO_SCORE.xlsx file to run Part B.")
scored_path = scored_files[0]
scored = pd.read_excel(scored_path, sheet_name="Evaluation", dtype={"Q_ID": str}).fillna("")

missing = scored[(scored["RAG_Verdict_S1"] == "") | (scored["NoRAG_Verdict"] == "")]
if len(missing):
    raise ValueError(f"Unscored rows: {list(missing['Q_ID'])}")
TWO_SCORERS = bool((scored["RAG_Verdict_S2"] != "").all())
if not TWO_SCORERS and (scored["RAG_Verdict_S2"] != "").any():
    raise ValueError("Scorer 2 filled only some rows: fill all 20 or leave the whole column empty.")
print("Scoring mode:", "two independent scorers" if TWO_SCORERS else "one scorer (agreement not computed)")

if TWO_SCORERS:
    scored["Scorers_Agree"] = scored["RAG_Verdict_S1"] == scored["RAG_Verdict_S2"]
else:
    scored["Scorers_Agree"] = True
need_final = scored[~scored["Scorers_Agree"] & (scored["RAG_Final_Verdict"] == "")]
if len(need_final):
    raise ValueError(f"Scorers disagree but no final verdict: {list(need_final['Q_ID'])}")
scored["RAG_Verdict"] = scored.apply(lambda r: r["RAG_Verdict_S1"] if r["Scorers_Agree"] else r["RAG_Final_Verdict"], axis=1)
scored["RAG_Score"] = scored["RAG_Verdict"].map(SCORE_OF)
scored["NoRAG_Score"] = scored["NoRAG_Verdict"].map(SCORE_OF)
bad = scored[scored["RAG_Score"].isna() | scored["NoRAG_Score"].isna()]
if len(bad):
    raise ValueError(f"Unknown verdict text in rows: {list(bad['Q_ID'])}")

A = scored[scored["Type"] == "Answerable"]
U = scored[scored["Type"] == "Unanswerable"]
rows = [
    ("Answerable questions", len(A), len(A)),
    ("Mean score on answerable questions (0 to 1)", round(A["RAG_Score"].mean(), 3), round(A["NoRAG_Score"].mean(), 3)),
    ("Correct answers (answerable)", int((A["RAG_Verdict"] == "Correct").sum()), int((A["NoRAG_Verdict"] == "Correct").sum())),
    ("Partly correct answers (answerable)", int((A["RAG_Verdict"] == "Partly correct").sum()), int((A["NoRAG_Verdict"] == "Partly correct").sum())),
    ("Wrong refusals (answerable)", int((A["RAG_Verdict"] == "Refused").sum()), int((A["NoRAG_Verdict"] == "Refused").sum())),
    ("Unanswerable questions", len(U), len(U)),
    ("Correct-refusal rate (unanswerable)", round(U["RAG_Score"].mean(), 3), round(U["NoRAG_Score"].mean(), 3)),
    ("Hallucinated answers (unanswerable)", int((U["RAG_Verdict"] == "Hallucinated").sum()), int((U["NoRAG_Verdict"] == "Hallucinated").sum())),
    ("Overall mean score (20 questions)", round(scored["RAG_Score"].mean(), 3), round(scored["NoRAG_Score"].mean(), 3)),
    ("Groundedness rate of RAG answers (answerable, Y)", round((A["RAG_Grounded"] == "Y").mean(), 3), "n/a"),
    ("Scorer agreement on RAG verdicts", round(scored["Scorers_Agree"].mean(), 3) if TWO_SCORERS else "n/a (one scorer)", "n/a"),
    ("Mean Hit@3 (answerable)", round(A["hit_at_k"].astype(float).mean(), 3), "n/a"),
    ("Mean Recall@3 (answerable)", round(A["recall_at_k"].astype(float).mean(), 3), "n/a"),
    ("MRR (answerable)", round(A["reciprocal_rank"].astype(float).mean(), 3), "n/a"),
    ("Mean response time, s", round(scored["total_s"].astype(float).mean(), 2), round(scored["norag_generation_s"].astype(float).mean(), 2)),
]
summary_df = pd.DataFrame(rows, columns=["Measure", "RAG chatbot", "No-RAG baseline"])
by_diff = (A.groupby("Difficulty")[["RAG_Score", "NoRAG_Score"]].mean().round(3)
           .reindex(["Easy", "Average", "Difficult"]).reset_index())
disagreements = scored[~scored["Scorers_Agree"]][["Q_ID", "RAG_Verdict_S1", "RAG_Verdict_S2", "RAG_Final_Verdict", "Scorer_Notes"]]

FINAL = OUT_DIR / f"{GROUP_NAME}_Final_Chatbot_Evaluation.xlsx"
with pd.ExcelWriter(FINAL, engine="openpyxl") as xw:
    summary_df.to_excel(xw, sheet_name="Summary", index=False)
    by_diff.to_excel(xw, sheet_name="By_Difficulty", index=False)
    scored.to_excel(xw, sheet_name="Evaluation", index=False)
    disagreements.to_excel(xw, sheet_name="Scorer_Disagreements", index=False)
    pd.read_excel(scored_path, sheet_name="Retrieved_Chunks").to_excel(xw, sheet_name="Retrieved_Chunks", index=False)
    pd.DataFrame(RUBRIC).to_excel(xw, sheet_name="Rubric", index=False)
scored.to_csv(OUT_DIR / f"{GROUP_NAME}_Final_Chatbot_Evaluation.csv", index=False)
files.download(str(FINAL))
files.download(str(OUT_DIR / f"{GROUP_NAME}_Final_Chatbot_Evaluation.csv"))
display(summary_df)
display(by_diff)
display(disagreements)

Saving TripleNeurons_Final_Chatbot_Evaluation_TO_SCORE.xlsx to TripleNeurons_Final_Chatbot_Evaluation_TO_SCORE (1).xlsx
Scoring mode: two independent scorers


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Measure,RAG chatbot,No-RAG baseline
0,Answerable questions,15.000,15
1,Mean score on answerable questions (0 to 1),0.567,0.3
2,Correct answers (answerable),4.000,0
3,Partly correct answers (answerable),9.000,9
4,Wrong refusals (answerable),0.000,0
5,Unanswerable questions,5.000,5
6,Correct-refusal rate (unanswerable),0.600,0.4
7,Hallucinated answers (unanswerable),2.000,3
8,Overall mean score (20 questions),0.575,0.325
9,"Groundedness rate of RAG answers (answerable, Y)",0.800,n/a


,Difficulty,RAG_Score,NoRAG_Score
0,Easy,0.4,0.2
1,Average,0.7,0.3
2,Difficult,0.6,0.4


,Q_ID,RAG_Verdict_S1,RAG_Verdict_S2,RAG_Final_Verdict,Scorer_Notes
0,E1,Partly correct,Correct,Partly correct,All 4 labels correct but cites “Article 3” ins...
1,E2,Incorrect,Partly correct,Incorrect,Gives only 1 of the 4 waste types (recyclables).
2,E3,Partly correct,Incorrect,Partly correct,"Covered container, no plastic bag; missing “dr..."
3,E4,Partly correct,Incorrect,Partly correct,Correct content but cites Guidebook p.20; the ...
4,E5,Partly correct,Correct,Partly correct,Only short-term hazards; answer cut off before...
5,A1,Correct,Partly correct,Correct,Complete list of special wastes.
7,A3,Correct,Partly correct,Correct,"Complete, including human waste, dead animals ..."
13,D4,Partly correct,Correct,Partly correct,“At home” is correct but one line only; no Sec...
